In [2]:

from mGST.low_level_jit import contract_mps_all_povm, cost_function_jax_mps
from mGST.additional_fns import perturbed_target_init, sampled_measurements, random_seq_design

from mGST.utility_functions_comparisons import get_compressed_rep_from_mgst_output, get_compressed_perturbed_rep_from_mgst, create_2q_emerald_gst_config

from mGST.algorithm import run_mGST

from mGST.trust_region import run_riemannian_optimization
from mGST.typing import TrustRegionOptions, GradientDescentOptions, OperatorSchedule, OptimizationScheduleItem
from mGST.visualization import plot_cost_function, plot_alternating_optimization
import jax.numpy as jnp
import numpy as np
import pickle

backend = "iqmfakeapollo"

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Setup

In [ ]:
num_qubits = 2
dim = 2**num_qubits
kraus_rank = dim**2
state_rank = dim
povm_rank = dim

Q2_GST = GSTConfiguration(
    qubit_layouts=[[0, 1]],
    gate_set="2QXYCZ",
    num_circuits=800,
    shots=1000,
    rank=kraus_rank,
)

kraus_tensor_mgst, kraus_mgst, povm_mgst, state_mgst, prob_matrix, indices_list = get_full_mgst_parameters_from_configuration(
    Q2_GST, backend, only_jax_variables=True
)

2026-01-21 12:59:31,786 - iqm.benchmarks.logging_config - INFO - Now generating 800 random GST circuits...
2026-01-21 12:59:32,214 - iqm.benchmarks.logging_config - INFO - Will transpile all 800 circuits according to fixed physical layout
2026-01-21 12:59:32,214 - iqm.benchmarks.logging_config - INFO - Transpiling for backend IQMFakeApolloBackend with optimization level 0, sabre routing method all circuits
2026-01-21 12:59:33,855 - iqm.benchmarks.logging_config - INFO - Submitting batch with 800 circuits corresponding to qubits [0, 1]
2026-01-21 12:59:33,864 - iqm.benchmarks.logging_config - INFO - Now executing the corresponding circuit batch
2026-01-21 12:59:33,920 - iqm.benchmarks.logging_config - INFO - Retrieving all counts
INFO:2026-01-21 12:59:37,995:jax._src.xla_bridge:927: Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
2026-01-21 12:59:37,995 - jax._src.xla_bridge - INFO - Unable to initialize backend 'rocm': module 'ja

In [ ]:
kraus_tensor_jax, povm_psd_jax, state_psd_jax = get_compressed_rep_from_mgst_output(kraus_mgst, povm_mgst, state_mgst, kraus_rank=kraus_rank, state_rank=state_rank, povm_rank=povm_rank)

povm_psd_mgst = np.array(povm_psd_jax)
state_psd_mgst = np.array(state_psd_jax)